# Loading sql file

In [2]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
    --------------------------------------- 0.3/18.2 MB ? eta -:--:--
    --------------------------------------- 0.3/18.2 MB ? eta -:--:--
    --------------------------------------- 0.3/18.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.2 MB 360.2 kB/s eta 0:00:50
   - -------------------------------------- 0.5/18.2 MB 360.2 kB/s eta 0:00:50
   - -------------------------------------- 0.5/18.2 MB 360.2 kB/s eta 0:00:50
   - -------------------------------------- 0.5/18.2 MB 360.2 kB/s eta 0:00:50
   - -------------------------------------- 0.5/18.2 MB 360.2 kB/s eta 0:00:50
   - -------------------------------------- 0


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install pymysql


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from urllib.parse import quote_plus
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

user = "root"
password = quote_plus("Sweety@123")
host = "localhost"
port = 3306
db_name = "employee_analytics_db"

server_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")
with server_engine.begin() as conn:
  conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {db_name}"))
  conn.execute(text(f"USE {db_name}"))
  conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
  conn.execute(text("DROP TABLE IF EXISTS fact_employee"))
  conn.execute(text("DROP TABLE IF EXISTS dim_department"))
  conn.execute(text("DROP TABLE IF EXISTS dim_city"))
  conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))

  conn.execute(text("""
        CREATE TABLE dim_department (
            dept_id INT PRIMARY KEY,
            dept_name VARCHAR(50) NOT NULL UNIQUE
        )
    """))
  conn.execute(text("""
        CREATE TABLE dim_city (
            city_id INT PRIMARY KEY,
            city_name VARCHAR(50) NOT NULL UNIQUE
        )
    """))
  conn.execute(text("""
        CREATE TABLE fact_employee (
            emp_record_id INT PRIMARY KEY,
            customer_id VARCHAR(20) NOT NULL,
            full_name VARCHAR(120) NOT NULL,
            dept_id INT NOT NULL,
            city_id INT NOT NULL,
            join_date DATE NOT NULL,
            tenure_years DECIMAL(5,2) NOT NULL,
            age INT NOT NULL CHECK (age BETWEEN 18 AND 65),
            salary INT NOT NULL CHECK (salary BETWEEN 10000 AND 500000),
            performance_rating INT NOT NULL CHECK (performance_rating BETWEEN 1 AND 5),
            is_duplicate_customer_id TINYINT(1) NOT NULL DEFAULT 0,
            CONSTRAINT fk_emp_dept FOREIGN KEY (dept_id) REFERENCES dim_department(dept_id),
            CONSTRAINT fk_emp_city FOREIGN KEY (city_id) REFERENCES dim_city(city_id)
        )
    """))

# Load cleaned Excel data and prepare normalized tables
df = pd.read_excel("Cleaned_Employee_Dataset_10000.xlsx")
df["Department_Clean"] = df["Department_Clean"].replace({"It": "IT", "Hr": "HR"})
df["Join_Date_Clean"] = pd.to_datetime(df["Join_Date_Clean"])

ref_date = pd.Timestamp("2026-11-21")
df["Tenure_Years"] = ((ref_date - df["Join_Date_Clean"]).dt.days / 365.25).round(2)
df["Full_Name_Clean"] = (
    df["First_Name_Clean"].astype(str) + " " + df["Last_Name_Clean"].astype(str)
)

dim_department = (
    df[["Department_Clean"]]
    .drop_duplicates()
    .sort_values("Department_Clean")
    .reset_index(drop=True)
)
dim_department["dept_id"] = dim_department.index + 1
dim_department = dim_department[["dept_id", "Department_Clean"]].rename(
    columns={"Department_Clean": "dept_name"}
)

dim_city = (
    df[["City_Clean"]]
    .drop_duplicates()
    .sort_values("City_Clean")
    .reset_index(drop=True)
)
dim_city["city_id"] = dim_city.index + 1
dim_city = dim_city[["city_id", "City_Clean"]].rename(
    columns={"City_Clean": "city_name"}
)

fact_emp = df.merge(
    dim_department, left_on="Department_Clean", right_on="dept_name", how="left"
).merge(dim_city, left_on="City_Clean", right_on="city_name", how="left")

fact_emp.insert(0, "emp_record_id", np.arange(1, len(fact_emp) + 1))
fact_emp["join_date"] = fact_emp["Join_Date_Clean"].dt.strftime("%Y-%m-%d")
fact_emp["is_duplicate_customer_id"] = fact_emp[
    "Is_Duplicate_Customer_ID"
].astype(int)

fact_sql = fact_emp[[
    "emp_record_id",
    "Customer_ID",
    "Full_Name_Clean",
    "dept_id",
    "city_id",
    "join_date",
    "Tenure_Years",
    "Age_Clean",
    "Salary_Clean",
    "Performance_Rating_Clean",
    "is_duplicate_customer_id",
]].rename(
    columns={
        "Customer_ID": "customer_id",
        "Full_Name_Clean": "full_name",
        "Tenure_Years": "tenure_years",
        "Age_Clean": "age",
        "Salary_Clean": "salary",
        "Performance_Rating_Clean": "performance_rating",
    }
)

db_engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}"
)
dim_department.to_sql(
    "dim_department", con=db_engine, if_exists="append", index=False
)
dim_city.to_sql("dim_city", con=db_engine, if_exists="append", index=False)
fact_sql.to_sql(
    "fact_employee",
    con=db_engine,
    if_exists="append",
    index=False,
    chunksize=1000,
)

with db_engine.connect() as conn:
  for tbl in ["dim_department", "dim_city", "fact_employee"]:
    cnt = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
    print(f"{tbl}: {cnt} rows")

dim_department: 6 rows
dim_city: 8 rows
fact_employee: 10000 rows


### To check error!

In [4]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

user = "root"
password = quote_plus("Sweety@123")
host = "localhost"
port = 3306

try:
  engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")
  with engine.connect() as conn:
    print("MySQL Version:", conn.execute(text("SELECT VERSION()")).scalar())
except Exception as e:
  print("Exact Error:", repr(e))

MySQL Version: 8.0.45
